In [2]:
import pandas as pd
import numpy as np

In [7]:


# ================== LOAD THE TWO FILES ==================
df1= pd.read_csv('./Test_Data/4K_MR_2051_EN_101_Jianan.csv')
df2= pd.read_csv('./Test_Data/4K_MR_2051_EN_101.csv')

# Basic info
print("Total rows:", len(df1))
print("Columns identical?", df1.columns.equals(df2.columns))

id_cols = ['SID', 'ISO_TIME', 'LAT', 'LON', 'PRES', 'VMAX']
print("Track information identical?", df1[id_cols].equals(df2[id_cols]))

# ================== SELECT METEOROLOGICAL COLUMNS ==================
var_cols = [col for col in df1.columns if col.startswith(('RH_', 'SD_'))]

# ================== FIND ALL DIFFERENCES ==================
# Use isclose to handle floating-point precision and NaNs
diff_mask = ~np.isclose(df1[var_cols].values, 
                        df2[var_cols].values, 
                        equal_nan=True, 
                        rtol=1e-8, 
                        atol=1e-10)

total_diff_cells = diff_mask.sum()
diff_row_indices = np.unique(np.where(diff_mask)[0])

print(f"\nTotal differing cells: {total_diff_cells}")
print(f"Rows with differences: {len(diff_row_indices)} ({len(diff_row_indices)/len(df1)*100:.2f}%)")

# ================== SUMMARY BY PRESSURE LEVEL ==================
print("\n=== Differences by Pressure Level ===")
levels = ['1000', '925', '850', '700', '600', '500', '400', '300', '250', '200', '150', '100']

for lev in levels:
    rh_col = f'RH_{lev}hPa'
    sd_col = f'SD_{lev}hPa'
    if rh_col in var_cols and sd_col in var_cols:
        rh_diff = (~np.isclose(df1[rh_col], df2[rh_col], equal_nan=True, rtol=1e-8, atol=1e-10)).sum()
        sd_diff = (~np.isclose(df1[sd_col], df2[sd_col], equal_nan=True, rtol=1e-8, atol=1e-10)).sum()
        print(f"{lev}hPa :  RH differs in {rh_diff:4d} rows,  SD differs in {sd_diff:4d} rows")

# ================== DETAILED DIFFERENCE TABLE ==================
diff_records = []
for r in diff_row_indices[:30]:   # Change 30 to None or 100 to see more
    for col in var_cols:
        v1 = df1.loc[r, col]
        v2 = df2.loc[r, col]
        if not np.isclose(v1, v2, equal_nan=True, rtol=1e-8, atol=1e-10):
            diff_records.append({
                'row_index': r,
                'ISO_TIME': df1.loc[r, 'ISO_TIME'],
                'LAT': round(df1.loc[r, 'LAT'], 2),
                'LON': round(df1.loc[r, 'LON'], 2),
                'Variable': col,
                'Value_Jianan': v1,
                'Value_Yohei': v2,
                'Abs_diff': abs(v1 - v2) if pd.notna(v1) and pd.notna(v2) else np.nan
            })

diff_table = pd.DataFrame(diff_records)
print("\n=== First 30 differing values (example) ===")
print(diff_table.head(30).to_string(index=False))

# Save full report
diff_table.to_csv('detailed_differences_Me_vs_Collaborator.csv', index=False)
print("\nFull detailed differences saved to: detailed_differences_Me_vs_Collaborator.csv")

Total rows: 841
Columns identical? True
Track information identical? True

Total differing cells: 1582
Rows with differences: 701 (83.35%)

=== Differences by Pressure Level ===
1000hPa :  RH differs in  701 rows,  SD differs in  701 rows
925hPa :  RH differs in   78 rows,  SD differs in   78 rows
850hPa :  RH differs in   10 rows,  SD differs in   10 rows
700hPa :  RH differs in    1 rows,  SD differs in    1 rows
600hPa :  RH differs in    1 rows,  SD differs in    1 rows
500hPa :  RH differs in    0 rows,  SD differs in    0 rows
400hPa :  RH differs in    0 rows,  SD differs in    0 rows
300hPa :  RH differs in    0 rows,  SD differs in    0 rows
250hPa :  RH differs in    0 rows,  SD differs in    0 rows
200hPa :  RH differs in    0 rows,  SD differs in    0 rows
150hPa :  RH differs in    0 rows,  SD differs in    0 rows
100hPa :  RH differs in    0 rows,  SD differs in    0 rows

=== First 30 differing values (example) ===
 row_index            ISO_TIME    LAT    LON   Variable 